# Pelatihan Model AI Rongsok (Google Colab)

Notebook ini disiapkan agar dapat dijalankan melalui browser HP Android. Gunakan runtime gratis bila tersedia. Notebook mengunduh dataset publik beranotasi dari Roboflow Universe, menyatukan label menjadi `kertas`, `plastik`, `besi`, `tembaga`, melatih YOLOv8n, mengevaluasi hasil, lalu mencoba ekspor TensorFlow.js.

## Batasan penting
- Tidak ada model terlatih yang disertakan atau diklaim selesai sebelum sel pelatihan dan ekspor benar-benar berhasil dijalankan.
- Roboflow API key dipakai hanya di runtime notebook dan diminta secara tersembunyi. Jangan tempelkan key ke kode atau repository. Dataset publik dapat memerlukan akun/API key gratis; jangan aktifkan paket berbayar.
- Dataset dasar memakai label `metal` sebagai pendekatan awal untuk kategori `besi`; label metal tidak menjamin objek benar-benar besi/baja. Harga harus tetap dikonfirmasi manual.
- Kelas `tembaga` diambil dari dataset terpisah yang lebih kecil. Evaluasi per kelas wajib dibaca, dan jangan gunakan deteksi tembaga sebagai dasar pembayaran otomatis tanpa pengujian lapangan.
- Ekspor YOLO TensorFlow.js tidak otomatis cocok dengan parser aplikasi saat ini. File hasil ekspor harus diperiksa dan parser harus disesuaikan dengan output nyata sebelum model diaktifkan di aplikasi.

Sumber dataset: [Waste Classification 4Class](https://universe.roboflow.com/swabdemo/waste-classification-4class) dan [Scrap Materials](https://universe.roboflow.com/janrells-workspace/scrap-materials). Periksa kembali ketentuan lisensi di halaman dataset sebelum distribusi model.

## 1. Siapkan runtime
Di Colab: **Runtime → Change runtime type → T4 GPU** jika tersedia. GPU gratis tidak dijamin. CPU tetap dapat dicoba, tetapi pelatihan bisa jauh lebih lambat.

In [ ]:
!pip -q install ultralytics roboflow pyyaml tensorflowjs

import os, json, shutil, zipfile, random, getpass
from pathlib import Path
import yaml
import torch
from roboflow import Roboflow
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
print('PyTorch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))

## 2. Unduh dua dataset publik
API key dimasukkan secara tersembunyi. Buat/ambil API key gratis di akun Roboflow jika diminta. Jangan pernah simpan key di notebook yang dibagikan.

In [ ]:
api_key = getpass.getpass('Roboflow API key (input tersembunyi): ').strip()
if not api_key:
    raise ValueError('API key kosong. Notebook dihentikan agar tidak melanjutkan dengan unduhan yang gagal.')

rf = Roboflow(api_key=api_key)
root = Path('/content/rongsok_training')
raw = root / 'raw'
if root.exists(): shutil.rmtree(root)
raw.mkdir(parents=True, exist_ok=True)

# Dataset utama: kelas paper, plastic, metal, dan kelas lain yang akan diabaikan.
base_project = rf.workspace('swabdemo').project('waste-classification-4class')
base_dataset = base_project.version(1).download('yolov8', location=str(raw / 'waste'))

# Dataset tambahan untuk contoh objek copper. Kelas lain dari dataset ini tidak dipakai.
copper_project = rf.workspace('janrells-workspace').project('scrap-materials')
copper_dataset = copper_project.version(1).download('yolov8', location=str(raw / 'scrap'))

print('Dataset dasar:', base_dataset.location)
print('Dataset tembaga:', copper_dataset.location)

## 3. Periksa kelas dan gabungkan anotasi
Sel ini tidak mengasumsikan indeks kelas dari dataset sumber. Ia membaca `data.yaml`, memetakan nama kelas, lalu menyalin anotasi YOLO yang relevan saja. Jika nama kelas tidak cocok, sel akan berhenti agar pemetaan tidak diam-diam salah.

In [ ]:
TARGET_NAMES = ['kertas', 'plastik', 'besi', 'tembaga']
TARGET_IDS = {name: i for i, name in enumerate(TARGET_NAMES)}
SPLITS = {'train': 'train', 'valid': 'valid', 'val': 'valid', 'test': 'test'}
merged = root / 'merged'
for split in ['train', 'valid', 'test']:
    (merged / split / 'images').mkdir(parents=True, exist_ok=True)
    (merged / split / 'labels').mkdir(parents=True, exist_ok=True)

def load_names(dataset_dir):
    config_path = Path(dataset_dir) / 'data.yaml'
    if not config_path.exists():
        raise FileNotFoundError(f'data.yaml tidak ditemukan: {config_path}')
    cfg = yaml.safe_load(config_path.read_text())
    names = cfg.get('names')
    if isinstance(names, dict): names = [names[k] for k in sorted(names, key=lambda x: int(x))]
    if not isinstance(names, list): raise ValueError(f'Format names tidak dikenal pada {config_path}: {names}')
    return [str(n).strip().lower().replace('_', ' ').replace('-', ' ') for n in names]

def remap_dataset(dataset_dir, source_tag, mapping):
    dataset_dir = Path(dataset_dir)
    names = load_names(dataset_dir)
    print(source_tag, 'kelas sumber:', names)
    name_to_target = {k.lower(): v for k, v in mapping.items()}
    unknown = [k for k in name_to_target if k not in names]
    if unknown: raise ValueError(f'Kelas yang diharapkan tidak ada pada {source_tag}: {unknown}. Kelas tersedia: {names}')
    source_id_to_target = {i: TARGET_IDS[name_to_target[name]] for i, name in enumerate(names) if name in name_to_target}
    copied = 0
    for split_dir in dataset_dir.iterdir():
        if not split_dir.is_dir(): continue
        split_key = SPLITS.get(split_dir.name.lower())
        if split_key is None: continue
        image_dir = split_dir / 'images'
        label_dir = split_dir / 'labels'
        if not image_dir.exists() or not label_dir.exists(): continue
        for image_path in image_dir.iterdir():
            if image_path.suffix.lower() not in {'.jpg', '.jpeg', '.png', '.webp'}: continue
            label_path = label_dir / f'{image_path.stem}.txt'
            if not label_path.exists(): continue
            remapped = []
            for line in label_path.read_text().splitlines():
                parts = line.strip().split()
                if len(parts) != 5: continue
                try: old_id = int(float(parts[0]))
                except ValueError: continue
                if old_id in source_id_to_target:
                    remapped.append(' '.join([str(source_id_to_target[old_id]), *parts[1:]]))
            if not remapped: continue
            dest_name = f'{source_tag}_{split_key}_{image_path.name}'
            shutil.copy2(image_path, merged / split_key / 'images' / dest_name)
            (merged / split_key / 'labels' / f'{Path(dest_name).stem}.txt').write_text('\n'.join(remapped) + '\n')
            copied += 1
    print(source_tag, 'gambar beranotasi relevan disalin:', copied)
    return copied

base_count = remap_dataset(base_dataset.location, 'base', {'paper': 'kertas', 'plastic': 'plastik', 'metal': 'besi'})
copper_count = remap_dataset(copper_dataset.location, 'copper', {'copper': 'tembaga'})
if base_count == 0 or copper_count == 0:
    raise RuntimeError('Dataset tidak menghasilkan anotasi relevan. Periksa versi, nama kelas, dan format unduhan sebelum melanjutkan.')

class_counts = {name: 0 for name in TARGET_NAMES}
split_class_counts = {split: {name: 0 for name in TARGET_NAMES} for split in ['train', 'valid', 'test']}
for split in ['train', 'valid', 'test']:
    for label in (merged / split / 'labels').glob('*.txt'):
        for line in label.read_text().splitlines():
            if line.strip():
                class_id = int(line.split()[0])
                class_name = TARGET_NAMES[class_id]
                class_counts[class_name] += 1
                split_class_counts[split][class_name] += 1
print('Jumlah anotasi per kelas:', class_counts)
print('Jumlah anotasi per kelas per split:', json.dumps(split_class_counts, indent=2, ensure_ascii=False))
if any(class_counts[name] == 0 for name in TARGET_NAMES):
    raise RuntimeError('Satu atau lebih kelas tidak memiliki anotasi. Jangan latih model dengan label kelas yang kosong.')
if any(split_class_counts['train'][name] == 0 for name in TARGET_NAMES):
    raise RuntimeError('Satu atau lebih kelas tidak ada di split train. Perlu dataset yang lebih cocok sebelum pelatihan.')
if not any(split_class_counts['valid'][name] > 0 for name in TARGET_NAMES):
    raise RuntimeError('Split valid kosong; evaluasi validasi tidak dapat dilakukan.')

data_yaml = {
    'path': str(merged),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'names': {i: name for i, name in enumerate(TARGET_NAMES)},
}
(merged / 'data.yaml').write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
print((merged / 'data.yaml').read_text())

## 4. Latih model ringan
Konfigurasi awal memakai YOLOv8n, ukuran gambar 320, dan maksimal 30 epoch agar lebih realistis untuk Colab gratis. Jika runtime berhenti karena kuota, jalankan ulang dari awal atau gunakan hasil yang sudah tersimpan di sesi. Jangan menilai model hanya dari training loss.

In [ ]:
device = 0 if torch.cuda.is_available() else 'cpu'
model = YOLO('yolov8n.pt')
train_result = model.train(
    data=str(merged / 'data.yaml'),
    imgsz=320,
    epochs=30,
    patience=8,
    batch=8 if torch.cuda.is_available() else 2,
    workers=2,
    device=device,
    seed=SEED,
    project=str(root / 'runs'),
    name='rongsok-yolov8n',
    exist_ok=True,
    plots=True,
    cache=False,
)
best_path = Path(train_result.save_dir) / 'weights' / 'best.pt'
if not best_path.exists(): raise FileNotFoundError(f'Bobot terbaik tidak ditemukan: {best_path}')
print('Bobot hasil pelatihan:', best_path)

## 5. Evaluasi dan simpan metrik
Periksa hasil per kelas, terutama `tembaga`. Dataset tembaga yang sedikit atau berbeda dari barang rongsok nyata dapat membuat metrik tampak bagus tetapi generalisasi buruk.

In [ ]:
trained = YOLO(str(best_path))
metrics = trained.val(data=str(merged / 'data.yaml'), split='test', imgsz=320, device=device, plots=True)
metric_summary = {
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'class_names': TARGET_NAMES,
    'class_annotation_counts_all_splits': class_counts,
}
(root / 'metrics.json').write_text(json.dumps(metric_summary, indent=2, ensure_ascii=False))
print(json.dumps(metric_summary, indent=2, ensure_ascii=False))
print('Per-class metrics (lihat tabel/plot evaluasi di output Ultralytics).')

## 6. Ekspor TensorFlow.js dan paketkan artefak
Ekspor ini hanya percobaan awal. Jika TensorFlow.js converter gagal, catat error-nya dan jangan menyalin file setengah jadi ke aplikasi. Hasil YOLO TFJS biasanya membutuhkan parser output khusus, bukan parser boxes/scores/classes terpisah yang saat ini ada di aplikasi.

In [ ]:
exported = trained.export(format='tfjs', imgsz=320)
exported_path = Path(exported)
print('Folder hasil ekspor:', exported_path)
model_jsons = list(exported_path.rglob('model.json')) if exported_path.exists() else []
if not model_jsons:
    raise FileNotFoundError('Ekspor tidak menghasilkan model.json. Jangan lanjutkan integrasi.')
for model_json in model_jsons:
    print('model.json:', model_json)
    model_info = json.loads(model_json.read_text())
    print('format model:', model_info.get('format'))
    print('weight manifest tersedia:', bool(model_info.get('weightsManifest')))

artifact = root / 'rongsok-tfjs-export.zip'
with zipfile.ZipFile(artifact, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in exported_path.rglob('*'):
        if file.is_file(): zf.write(file, file.relative_to(exported_path.parent))
    zf.write(root / 'metrics.json', 'metrics.json')
print('Artefak:', artifact, 'ukuran MB:', round(artifact.stat().st_size / (1024*1024), 2))

## 7. Unduh artefak
Unduh ZIP hasil ekspor dan simpan di perangkat/Drive. Jangan unggah model ke `public/models/` atau aktifkan deteksi pada production sebelum output tensor, kelas, preprocessing, dan performa di HP diuji. Model YOLO hasil ekspor tidak dijamin cocok dengan parser yang sekarang.

In [ ]:
from google.colab import files
files.download(str(artifact))